# Celery + Redis 定时任务调度 —— 逐层演示

对照讲义：《Celery + Redis 定时任务调度讲义》

本 Notebook 完全按讲义章节走，每一层都是"先写散装代码跑通 → 观察现象 → 遇到问题 → 进入下一层"。
直到第 6 章才把散装代码收敛成 ReminderManager。

启动前提（三个终端窗口分别执行）：
```powershell
# 终端 1：Worker
uv run celery -A celery_reminder.celery_app:app worker -l INFO -Q reminders --pool solo
# 终端 2：Beat（RedBeat Scheduler）
uv run celery -A celery_reminder.celery_app:app beat -l INFO
# 终端 3（可选）：Agent 事件消费器
uv run python -m celery_reminder.events
# 终端 4（可选）：Flower（可视化监控）
uv run celery -A celery_reminder.celery_app flower --port=5555
```



## 环境导入 & 自检

In [1]:

import redis

from celery_reminder.celery_app import app  # Celery 实例
from celery_reminder.tasks import deliver_reminder  # 注册名 = agent.reminder.deliver

for label, url in [("Broker(db0)", app.conf.broker_url),
                   ("State(db1)", app.conf.state_redis_url),
                   ("RedBeat(db2)", app.conf.redbeat_redis_url)]:
    print(f"[{label}] {redis.Redis.from_url(url).ping()}  ({url})")
print()
print(f"beat_scheduler       = {app.conf.beat_scheduler}")
print(f"beat_max_loop_interval = {app.conf.beat_max_loop_interval}s")
print(f"task name            = {deliver_reminder.name}")


[Broker(db0)] True  (redis://127.0.0.1:6379/0)
[State(db1)] True  (redis://127.0.0.1:6379/1)
[RedBeat(db2)] True  (redis://127.0.0.1:6379/2)

beat_scheduler       = redbeat.RedBeatScheduler
beat_max_loop_interval = 5s
task name            = agent.reminder.deliver



---

# 第 0 章 先分清四种「定时」

| 语义 | 业务说法 | Celery 实现 | 需要 beat 吗 | 触发次数 |
|---|---|---|---|---|
| 延时一次 | 30 分钟后提醒我 | `apply_async(countdown=N)` | 不需要 | 1 次 |
| 定点一次 | 今天 15:30 提醒我 | `apply_async(eta=带时区 datetime)` | 不需要 | 1 次 |
| 固定间隔 | 每 5 分钟检查一次 | `beat_schedule` 里 `schedule=300` | 需要 | 无限次 |
| 日历式 | 每天 9:00 | `beat_schedule` 里 `schedule=crontab(...)` | 需要 | 无限次 |

1. countdown / eta 是**延迟投递**，不是定时调度。没有周期。
2. **beat 是唯一的闹钟**。没有 beat 进程，任何周期性提醒都不会自己发生。
3. 定点和周期不要互相代替。要用「每天 9 点」就必须是 crontab。



---

# 第 3 章｜第 1 层 —— 一次性延时提醒

不需要 beat 进程 —— Redis 自己帮你挂闹钟。

## 3.1 countdown：N 秒后提醒一次


In [6]:

# ===== 讲义 3.1 节：直接 apply_async(countdown=N) =====
from celery_reminder.tasks import deliver_reminder

res = deliver_reminder.apply_async(
    args=[{"agent_id": "agent-001", "remind_id": "12345678901", "text": "3 秒到了！countdown 测试"}],
    countdown=3,
    queue="reminders",
)
print(f"task_id = {res.id}")
print("Worker 日志约 3 秒后输出 [DELIVERED]")


task_id = 092cfb0c-bee4-415e-839b-0eadee4edd23
Worker 日志约 3 秒后输出 [DELIVERED]


## 3.2 eta：指定某个绝对时刻提醒一次

In [15]:

# ===== 讲义 3.2 节：eta 必须带时区！=====
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

at = datetime.now(tz=ZoneInfo("Asia/Shanghai")) + timedelta(seconds=5)
# at = datetime(2026, 11, 11, 16, 40, tzinfo=ZoneInfo("Asia/Shanghai"))

res = deliver_reminder.apply_async(
    args=[{"agent_id": "agent-002", "text": "eta 时刻到了！"}],
    eta=at,
    queue="reminders",
)
print(f"eta = {at.strftime('%H:%M:%S')}  task_id = {res.id}")
print()
print("三个必须记住的点：")
print("  1. eta 必须带时区（aware datetime），传 naive 会被当 UTC 解释，整体晚 8 小时")
print("  2. eta 是绝对时刻，不受 timezone 配置影响")
print("  3. eta 已过去时 Celery 立刻执行（不会报错不会跳过）")


eta = 17:26:16  task_id = a23e911c-bf58-427f-bc21-90b948dc0a2a

三个必须记住的点：
  1. eta 必须带时区（aware datetime），传 naive 会被当 UTC 解释，整体晚 8 小时
  2. eta 是绝对时刻，不受 timezone 配置影响
  3. eta 已过去时 Celery 立刻执行（不会报错不会跳过）


## 3.3 revoke：撤销尚未触发的提醒

In [5]:

# ===== 讲义 3.3 节：广播撤销 =====
res = deliver_reminder.apply_async(
    args=[{"agent_id": "agent-003", "text": "这条会被撤销，不会真的投递"}],
    countdown=3,
    queue="reminders",
)
task_id = res.id
print(f"已投 3s 后提醒  task_id={task_id}")

# 用户改主意了
app.control.revoke(task_id, terminate=False)
print(f"app.control.revoke({task_id}) 已调用")
print()
print("边界：revoke 只对还在队列里、尚未被 Worker 取走的任务有效")
print("Worker 重启后撤销名单丢失（默认存内存）")
print("实战推荐软撤销：任务体第一行查 cancel 标记就 return")


已投 3s 后提醒  task_id=4e02058c-974d-4252-ad68-17ce48dd5a33
app.control.revoke(4e02058c-974d-4252-ad68-17ce48dd5a33) 已调用

边界：revoke 只对还在队列里、尚未被 Worker 取走的任务有效
Worker 重启后撤销名单丢失（默认存内存）
实战推荐软撤销：任务体第一行查 cancel 标记就 return



---

# 第 5 章｜第 3 层 —— 动态周期提醒（RedBeat）

RedBeat 把定时表搬进 Redis，任何进程都能读写。beat 每次 tick 从 Redis 读表。

启动 beat 时必须指定 Scheduler：
```
celery -A ... beat -l INFO --scheduler redbeat.RedBeatScheduler
```
我们已在 celery_app.py 配了 beat_scheduler，所以不用敲。


## 5.1 新增一条周期提醒（讲义 5.4 节）

In [9]:

# ===== 讲义 5.4 新增 —— 固定间隔 =====
from redbeat import RedBeatSchedulerEntry

entry = RedBeatSchedulerEntry(
    name="remind-agent001-water",                     # 条目名，唯一；也是 Redis key 一部分
    task="agent.reminder.deliver",                     # 必须和 @app.task(name=...) 一致
    schedule=10,                                       # 秒；也接受 timedelta / crontab
    args=({"agent_id": "agent-001", "text": "每 10 秒喝水提醒"},),
    app=app,                                           # 只传 app 一个参数！
)
entry.save()   # 写入 Redis，beat 下次 tick 就能看到

print(f"✅ 新增固定间隔：{entry.name}  每 10s")


✅ 新增固定间隔：remind-agent001-water  每 10s


In [17]:

# ===== 讲义 5.4 新增 —— 日历式 crontab =====
from celery.schedules import crontab

entry2 = RedBeatSchedulerEntry(
    name="remind-agent002-minute",
    task="agent.reminder.deliver",
    schedule=crontab(minute="*/15"),                  # 每 15 分钟
    args=({"agent_id": "agent-002", "text": "每 15 分钟巡检"},),
    app=app,
)
entry2.save()

print(f"✅ 新增 crontab：{entry2.name}  每 15 分钟")


✅ 新增 crontab：remind-agent002-minute  每 15 分钟


## 5.2 查看所有周期提醒（讲义 5.4 节「查看」）

In [13]:

# ===== 讲义 5.4 查看 —— 直接查 Redis ZSET =====
import redis

rds = redis.Redis.from_url(app.conf.redbeat_redis_url, decode_responses=True)
sched_key = app.conf.redbeat_key_prefix + ":schedule"   # 默认 redbeat::schedule
print(f"ZSET key = {sched_key}")
print()

for raw_key in rds.zrange(sched_key, 0, -1):
    e = RedBeatSchedulerEntry.from_key(raw_key, app=app)
    due, seconds_to_next = e.is_due()
    score = int(rds.zscore(sched_key, raw_key))
    print(f"name       = {e.name}")
    print(f"  task     = {e.task}")
    print(f"  schedule = {e.schedule}")
    print(f"  enabled  = {e.enabled}")
    print(f"  next_run = ~{int(seconds_to_next)}s  (ts={score})")
    print()


ZSET key = redbeat::schedule

name       = remind-agent001-water
  task     = agent.reminder.deliver
  schedule = <freq: 10.00 seconds>
  enabled  = False
  next_run = ~5s  (ts=1794388587)

name       = celery.backend_cleanup
  task     = celery.backend_cleanup
  schedule = <crontab: 0 4 * * * (m/h/dM/MY/d)>
  enabled  = True
  next_run = ~38602s  (ts=1794418340)



## 5.3 暂停一条

In [12]:

# ===== 讲义 5.4 暂停 —— enabled=False =====
key = RedBeatSchedulerEntry.generate_key(app, "remind-agent001-water")
entry = RedBeatSchedulerEntry.from_key(key, app=app)
entry.enabled = False
entry.save()

print(f"⏸️  已暂停 {entry.name}")
print("观察 Worker 日志：agent-001 不再触发")


⏸️  已暂停 remind-agent001-water
观察 Worker 日志：agent-001 不再触发


## 5.4 恢复 + 删除

In [14]:

# 恢复
key = RedBeatSchedulerEntry.generate_key(app, "remind-agent001-water")
entry = RedBeatSchedulerEntry.from_key(key, app=app)
entry.enabled = True
entry.save()
print(f"▶️  已恢复 {entry.name}")

# 删除 —— generate_key + from_key 必须配对！
for name in ["remind-agent001-water", "remind-agent002-minute"]:
    key = RedBeatSchedulerEntry.generate_key(app, name)
    try:
        RedBeatSchedulerEntry.from_key(key, app=app).delete()
        print(f"🗑️  已删除 {name}")
    except KeyError:
        print(f"  skip {name}（不存在）")


▶️  已恢复 remind-agent001-water
🗑️  已删除 remind-agent001-water
  skip remind-agent002-minute（不存在）



## 5.5 讲义里的五个高频错误

| 现象 | 原因 | 处理 |
|---|---|---|
| 加了条目但一直不跑 | beat 没带 --scheduler | 配 beat_scheduler |
| 加了条目要等好几分钟 | max_interval 默认 300s | beat_max_loop_interval=5 |
| 删除报 KeyError | 传了条目名而非完整 key | generate_key 拼前缀 |
| 同一条目两个环境互相打架 | 共用同一 Redis DB | 隔离 DB/实例 |
| beat 日志刷 lock timeout | 周期远大于 redbeat_lock_timeout | 调大或留空 |



---

# 第 7 章｜第 5 层 —— 可靠性与正确性

消息队列是 **at-least-once**，不是 exactly-once。Worker 重跑是常态。

**宁可重，不要丢**；然后用幂等把重的代价压到零。


## 7.1 投递侧幂等闸门（讲义 7.1）

In [23]:

# ===== 讲义 7.1 闸门 1 —— SET NX =====
import uuid, redis

_state = redis.Redis.from_url(app.conf.state_redis_url, decode_responses=True)
remind_id = uuid.uuid4().hex

def guard_delivery(remind_id):
    """返回 True 表示首次投递，可以继续。"""
    return bool(_state.set(f"remind:sent:{remind_id}", "1", nx=True, ex=86400))

print(f"第一次 guard_delivery: {guard_delivery(remind_id)}")   # True —— 闸门打开
print(f"第二次 guard_delivery: {guard_delivery(remind_id)}")   # False —— Worker 重跑也过不了
print(f"第三次 guard_delivery: {guard_delivery(remind_id)}")   # False
print(f"Redis TTL = {_state.ttl(f'remind:sent:{remind_id}')}s")


第一次 guard_delivery: True
第二次 guard_delivery: False
第三次 guard_delivery: False
Redis TTL = 86400s


## 7.2 Agent 互斥锁（讲义 7.2）

In [ ]:

# ===== 讲义 7.2 简单版锁：防同一 Agent 被并发唤醒 =====
import uuid

_state = redis.Redis.from_url(app.conf.state_redis_url, decode_responses=True)
agent_id = "agent-001"
lock_key = f"agent:lock:{agent_id}"

# Agent A 拿锁
token1 = uuid.uuid4().hex
got_A = _state.set(lock_key, token1, nx=True, ex=15)
print(f"Agent A 拿锁: {bool(got_A)}")    # True

# Agent B 同时来拿同一个锁
got_B = _state.set(lock_key, "some-token", nx=True, ex=15)
print(f"Agent B 拿锁: {bool(got_B)}")    # False —— 拿不到就放弃本轮

# Agent A 释放（检查 token 防止误删别人的锁）
if _state.get(lock_key) == token1:
    _state.delete(lock_key)
    print(f"Agent A 释放锁: OK")

got_B2 = _state.set(lock_key, "some-token", nx=True, ex=15)
print(f"Agent B 再拿锁: {bool(got_B2)}")  # True —— 锁已释放

_state.delete(lock_key)
print()
print("两把不同的锁，别混：")
print("  redbeat::lock  —— 防多个 beat 重复生成任务（调度层）")
print("  agent:lock:*   —— 防同一 Agent 被并发唤醒（执行层，必须自己加）")



---

# 第 8 章｜第 6 层 —— 架构：Worker 只做投递，Agent 自己消费

核心主张：
    Celery Worker  ->  XADD agent:events  ->  Agent 进程 XREADGROUP 消费
       （毫秒级）        （Redis Stream）        （Agent 自己的上下文）

为什么不在 Celery 任务里直接跑 Agent？
1. 状态不同步 —— Worker 里的 Agent 是另一个内存副本，用户侧看不到
2. 资源浪费 —— 为发一句提醒加载整个 Agent
3. 阻塞调度 —— 任务跑 3 分钟，队列积压
4. 不可控 —— Worker 扩缩容会顺手改变 Agent 并发度


## 8.1 直接写一条事件到 Stream

In [ ]:

# ===== 讲义 6.3 tasks.py —— 核心就是这一行 =====
import json, redis

_state = redis.Redis.from_url(app.conf.state_redis_url, decode_responses=True)
STREAM = "agent:events"

msg_id = _state.xadd(
    STREAM,
    {"data": json.dumps(
        {"type": "reminder", "payload": {
            "agent_id": "agent-999",
            "text": "来自 Notebook 直接写的事件",
            "remind_id": "demo-abc-123",
        }}, ensure_ascii=False)},
    maxlen=100_000,
    approximate=True,
)
print(f"✅ 已写事件  msg_id={msg_id}")
print(f"   agent:events 总长 = {_state.xlen(STREAM)}")

print("\n最近 5 条：")
for mid, fields in _state.xrevrange(STREAM, "+", "-", count=5):
    event = json.loads(fields["data"])
    p = event.get("payload", {})
    print(f"  [{mid}] agent={p.get('agent_id')}  text={p.get('text')}")


## 8.2 Agent 侧消费组（讲义 8.2 events.py）

In [ ]:

# ===== 讲义 8.2 consume_once —— 阻塞消费 =====
import json, redis

_state = redis.Redis.from_url(app.conf.state_redis_url, decode_responses=True)
STREAM, GROUP = "agent:events", "agents"

# 确保消费组存在（幂等）
try:
    _state.xgroup_create(STREAM, GROUP, id="0", mkstream=True)
    print(f"✅ 创建消费组 {GROUP}")
except redis.ResponseError:
    print(f"✅ 消费组 {GROUP} 已存在")

consumer = "notebook-consumer"
print(f"\n尝试消费一条（block 2s）...")

resp = _state.xreadgroup(GROUP, consumer, {STREAM: ">"}, count=1, block=2000)
if resp:
    for _, messages in resp:
        for msg_id, fields in messages:
            event = json.loads(fields["data"])
            p = event.get("payload", {})
            print(f"  ✅ 消费到  msg_id={msg_id}")
            print(f"     agent={p.get('agent_id')}")
            print(f"     text ={p.get('text')}")
            _state.xack(STREAM, GROUP, msg_id)
            print(f"     已 ack")
else:
    print("  （没有新消息）")

# 查看消费组状态
print(f"\n消费组状态：")
try:
    for g in _state.xinfo_groups(STREAM):
        print(f"  group={g['name']}  consumers={g['consumers']}  pending={g['pending']}")
except Exception:
    print("  暂无消费组信息")



## 8.3 分层职责表

| 层 | 职责 | 不该做的事 |
|---|---|---|
| Beat | 算时间、投递任务 | 不碰业务、不加载 Agent |
| Worker | 幂等判断 + 写事件流 | 不跑 Agent、不调模型 |
| Agent 进程 | 消费事件、发消息 | 不自己算时间、不自己轮询 |
| Redis | 消息 + 定时表 + 幂等键 | 不当缓存池 |



---

# 第 6 章｜第 4 层 —— 工程化封装（ReminderManager）

到这里你已经会写散装代码了。现在把它**收成业务侧只调三个方法的模块**。

这不是"更高级"，而是"散装代码里的常见模式被抽成了方法"：
- `once_in` 就是「apply_async(countdown=...) + 存 task_id 映射」
- `every` 就是「RedBeatSchedulerEntry(...) + save()」
- `cancel_once` 就是「查 task_id + revoke + 写软撤销标记」


In [ ]:

# ===== 讲义 6.4 —— ReminderManager =====
from celery_reminder.reminder import reminder   # 模块级单例

# ---- 一次性（对应第 3 章散装代码）----
rid = reminder.once_in(
    {"agent_id": "agent-001", "text": "用封装的 once_in 投一条"},
    delay_seconds=3,
)
print(f"once_in -> remind_id={rid}  (内部就是 apply_async(countdown=3))")

import time
time.sleep(4)
print("Worker 日志应该有 [DELIVERED]")

# ---- 周期（对应第 5 章散装代码）----
rid2 = reminder.every(
    {"agent_id": "agent-002", "text": "用封装的 every 投一条周期"},
    seconds=30,
)
print(f"\nevery -> remind_id={rid2}  (内部就是 RedBeatSchedulerEntry + save)")

time.sleep(2)
entries = reminder.list_every()
print(f"\nlist_every() -> {len(entries)} 条：")
for e in entries:
    print(f"  remind_id={e['id']}  kind={e['kind']}  schedule={e['schedule']}")

# 清理
reminder.cancel_every(rid2)
reminder.cancel_once(rid)
print("\n✅ 演示完，全部清理了")



---

## 附录 A：Redis 键一览（直接 redis-cli 可查）

```bash
redis-cli -n 0 ping      # Broker
redis-cli -n 1 ping      # State
redis-cli -n 2 ping      # RedBeat

redis-cli -n 2 zrange redbeat::schedule 0 -1 withscores
redis-cli -n 2 hget redbeat::remind-s-<rid> definition

redis-cli -n 1 xlen agent:events
redis-cli -n 1 xinfo groups agent:events

redis-cli -n 0 llen reminders
redis-cli -n 0 info stats | grep evicted_keys   # 必须为 0！
```

## 附录 B：什么时候不该用 Celery（讲义 9.2）

每天几十条提醒、单机跑、不想多维护两个进程 → 60 行 ZSET 自己搭：

```python
rds = redis.Redis.from_url("redis://127.0.0.1:6379/0")
rds.zadd("remind:zset", {payload_json: now + delay})
# 执行：while True: due = zrangebyscore(zset, -inf, now); zrem ...; handle()
```

Celery 白送：**重试 + 可观测 + 横向扩 Worker**。需要就用，不需要就自己搭。



---

# 业务数据层｜第 6.5 章 —— 创建任务怎么持久化？

你可能注意到了一个问题：ReminderManager 投出去的提醒，业务字段（"谁"、"提醒什么"、"什么时候"）藏在 RedBeat 条目的 args 和 Redis 的 task_id 映射里。Redis 重启（哪怕开了 AOF），业务侧想查"我的所有提醒"就没地方查。

**两条线索并行**：

  调度层（Redis）            业务层（SQLite）
  ──────────────────         ──────────────────
  Broker 队列（db0）          reminders 表
  RedBeat 定时表（db2）       ↑ 业务实体的结构化存储
  幂等键 / 事件流（db1）      ↑ 查询 / 过滤 / 统计的唯一入口

**创建顺序：先写 DB，再投调度**。DB 失败就不投，保持一致。


## 6.5.1 业务表 schema

In [ ]:

# SQLite 表结构（SQLite 零依赖，生产换 Postgres 只换 Driver）
schema = """
CREATE TABLE reminders (
    id          TEXT PRIMARY KEY,        -- remind_id (UUID hex)
    agent_id    TEXT NOT NULL,
    text        TEXT NOT NULL,
    kind        TEXT NOT NULL,           -- 'once' | 'periodic'
    schedule    TEXT NOT NULL,           -- JSON: {seconds:30} / {cron:"*/15..."} / {eta_iso:"..."}
    status      TEXT NOT NULL DEFAULT 'pending'
                CHECK (status IN ('pending', 'firing', 'done', 'cancelled')),
    created_at  TEXT NOT NULL,           -- ISO 8601 + timezone
    updated_at  TEXT NOT NULL,
    fired_at    TEXT                     -- 最近一次投递成功时间
);
"""
print(schema)
print("status 迁移:")
print("  创建  →  pending")
print("  Worker 闸门过  →  firing  + fired_at 记时间")
print("  投递成功:")
print("    once    →  done  (只触发一次)")
print("    periodic →  firing 保持  (下一个周期 beat 再来)")
print("  用户撤销  →  cancelled")


## 6.5.2 验证：创建 → DB 有记录

In [ ]:

from celery_reminder.reminder import reminder
from datetime import timedelta
from zoneinfo import ZoneInfo

# 清一下之前的（可选）
import sqlite3
c = sqlite3.connect("reminders.db")
c.execute("DELETE FROM reminders")
c.commit()
c.close()

print("=== 创建两个提醒，看业务表怎么记 ===")

# 1) countdown
rid1 = reminder.once_in(
    {"agent_id": "agent-X", "text": "DB 层验证 countdown"},
    delay_seconds=120,
)
print(f"1. countdown 提醒创建: {rid1}")
print(f"   DB 记录: {reminder.get(rid1)}")
print()

# 2) periodic
rid2 = reminder.every(
    {"agent_id": "agent-Y", "text": "DB 层验证 periodic"},
    seconds=60,
)
print(f"2. periodic 提醒创建: {rid2}")
print(f"   DB 记录: {reminder.get(rid2)}")
print()

# 3) 列表查询
print("=== 列表查询（从 DB 读，不是翻 RedBeat ZSET）===")
for r in reminder.list():
    print(f"  [{r['status']}] {r['kind']} {r['agent_id']}: {r['text']}")

# 4) 过滤查询
print()
print("=== 过滤：只看 pending ===")
for r in reminder.list(status="pending"):
    print(f"  {r['agent_id']}: {r['text']}")

print()
print("=== 过滤：agent-Y 的所有提醒 ===")
for r in reminder.list(agent_id="agent-Y"):
    print(f"  {r['kind']} {r['text']}")

# 5) 统计
print()
print("=== 统计 ===")
print(f"  总数={reminder.count()}  pending={reminder.count('pending')}  "
      f"firing={reminder.count('firing')}  cancelled={reminder.count('cancelled')}")


## 6.5.3 撤销 → DB 同步 cancelled

In [ ]:

print("=== 撤销 rid2（periodic），看 DB 怎么变 ===")
print(f"撤销前: {reminder.get(rid2)['status']}")

ok = reminder.cancel_every(rid2)
print(f"cancel_every → {ok}")

rec = reminder.get(rid2)
print(f"撤销后: {rec['status']}  (应该是 cancelled)")
print(f"RedBeat 条目也被删了（调度层 + 业务层同步撤销）")


## 6.5.4 Worker 投递 → DB 状态自动迁移（需要重启 Worker 进程才生效）

In [ ]:

# Worker 投递时会自动回写业务表状态：
#   闸门过 → firing + fired_at
#   投递成功 → once 变 done，periodic 保持 firing
#
# Worker 要重启才能加载新代码。重启后，投递过程中就能看到：
#   DB 记录的 status 从 pending → firing → done

print("Worker 投递状态迁移（重启 Worker 后生效）：")
print("  deliver_reminder 函数体里：")
print("    from celery_reminder.store import store as db  ← 延迟 import")
print("    db.update_status(remind_id, 'firing', fired_at=True)")
print("    ... 写事件流 ...")
print("    if kind == 'once':")
print("        db.update_status(remind_id, 'done')")
print()
print("为什么延迟 import store？")
print("  Worker 进程 fork 后，模块级 SQLite 连接会继承父进程 fd，会炸。")
print("  在函数体内 import，每次执行拿到的是新连接。")


## 6.5.5 重启恢复（进阶）

In [ ]:

print("=== Redis 重启丢了调度？扫描业务表重建 ===\n")

# 这是个运维脚本思路，不在 Notebook 里真执行：
#   业务表是唯一可信来源。Redis 丢了可以重建，DB 丢了就只能从备份恢复。

from celery_reminder.reminder import reminder
from datetime import datetime

# 1) 扫所有 pending 的 once，重新 apply_async
print("1) 重建一次性提醒：")
for r in reminder.list(kind="once", status="pending"):
    schedule = r["schedule"]
    if "eta_iso" in schedule:
        at = datetime.fromisoformat(schedule["eta_iso"])
        reminder.once_at(
            {"agent_id": r["agent_id"], "text": r["text"]},
            at=at, remind_id=r["id"],
        )
    elif "seconds" in schedule:
        reminder.once_in(
            {"agent_id": r["agent_id"], "text": r["text"]},
            delay_seconds=schedule["seconds"], remind_id=r["id"],
        )
    print(f"   + {r['id']} ({r['kind']}) 重建完成")

# 2) 扫所有 periodic 的，重新 save RedBeat 条目
print("\n2) 重建周期提醒：")
for r in reminder.list(kind="periodic", status="pending"):
    schedule = r["schedule"]
    if "seconds" in schedule:
        reminder.every(
            {"agent_id": r["agent_id"], "text": r["text"]},
            seconds=schedule["seconds"], remind_id=r["id"],
        )
    elif "cron" in schedule:
        print(f"   + {r['id']} cron 需要解析，略")
    print(f"   + {r['id']} ({r['kind']}) 重建完成")

print("\n✅ 调度层已从业务表重建完成")
print("   前提：业务表要有 WAL / 定时备份")



## 6.5.6 两条线索的关系

| 操作 | 业务层（SQLite） | 调度层（Redis） | 顺序 |
|---|---|---|---|
| once_in / once_at | INSERT status='pending' | apply_async(countdown/eta) | **先 DB，再投调度** |
| every | INSERT status='pending' | RedBeatSchedulerEntry.save() | **先 DB，再投调度** |
| cancel_once | UPDATE status='cancelled' | revoke + soft cancel | **先 DB，再撤销** |
| cancel_every | UPDATE status='cancelled' | RedBeat entry.delete() | **先 DB，再撤销** |
| Worker 投递 | UPDATE status='firing/done' | 写事件流 | 调度层触发 → Worker 回写业务层 |
| 查询 | SELECT ... WHERE ... | —— | **只从 DB 查** |

**为什么不是反过来（先投调度再写 DB）？**
如果 apply_async 成功了但 INSERT 失败，消息就丢了——Redis 里有一条没人知道业务内容的任务，DB 里也没记录。
**先写 DB 再投调度**是业务一致性的基本姿势。
